# Lab: HN CLI gate
Fetch, validate, emit. All 20 rows must pass the Story schema.

In [ ]:
!pip install -q httpx tenacity pydantic

In [ ]:
import asyncio, json, httpx
from typing import Optional
from pydantic import BaseModel, Field, ValidationError, HttpUrl
from tenacity import retry, stop_after_attempt, wait_exponential_jitter, retry_if_exception_type

HN = 'https://hacker-news.firebaseio.com/v0'

class Story(BaseModel):
    id: int
    by: str
    title: str = Field(min_length=1)
    score: int = Field(ge=0)
    time: int
    url: Optional[HttpUrl] = None
    descendants: Optional[int] = Field(default=None, ge=0)

@retry(retry=retry_if_exception_type((httpx.TransportError, httpx.HTTPStatusError)),
       wait=wait_exponential_jitter(initial=0.3, max=5),
       stop=stop_after_attempt(4), reraise=True)
async def _get(c, u):
    r = await c.get(u, timeout=10); r.raise_for_status(); return r.json()

async def fetch(limit=20):
    async with httpx.AsyncClient(limits=httpx.Limits(max_connections=20)) as c:
        ids = (await _get(c, f'{HN}/topstories.json'))[:limit*2]
        sem = asyncio.Semaphore(20)
        async def one(i):
            async with sem:
                return await _get(c, f'{HN}/item/{i}.json')
        return await asyncio.gather(*(one(i) for i in ids))

raw = await fetch(20)
valid = []
for item in raw:
    if not item: continue
    try:
        valid.append(Story.model_validate(item))
    except ValidationError:
        continue
    if len(valid) >= 20: break
print(json.dumps([s.model_dump(mode='json') for s in valid[:3]], indent=2))
print(f'\n{len(valid)}/20 validated')

No LLM call. cost = $0.00 / run.